In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("/content/drive/MyDrive/datasets bus sample/bus_sample.csv")

In [4]:
df.shape

(50, 5)

In [5]:
df.head(20)

,Datetime,DirectionRef,PublishedLineName,NextStopPointName,TimeToArrival
0,2017-06-30 22:03:02,0,0.582369,0.288095,0.400000
1,2017-06-20 13:45:15,0,1.067845,0.990972,1.033333
2,2017-06-08 17:36:45,0,1.089518,0.806250,0.516667
3,2017-06-25 09:23:24,0,0.683207,0.887121,0.666667
4,2017-06-09 18:48:30,1,0.875618,1.120000,1.450000
5,2017-06-02 17:55:26,0,0.849538,0.946296,1.200000
6,2017-06-28 15:29:33,0,0.695106,0.431081,0.516667
7,2017-06-26 13:25:21,0,0.633188,0.558621,0.383333
8,2017-06-20 06:54:45,0,2.514065,3.773381,1.600000
9,2017-06-01 15:13:52,1,0.789050,0.373333,0.716667


In [6]:
df.dtypes

,0
Datetime,object
DirectionRef,int64
PublishedLineName,float64
NextStopPointName,float64
TimeToArrival,float64


In [7]:
from sklearn.model_selection import train_test_split
train, test = train_test_split(df, test_size=0.2)
# Drop rows with any NaN values in the DataFrame
train = train.dropna()
test = test.dropna()

In [8]:
train['Datetime'] = pd.to_datetime(train['Datetime'], errors='coerce').astype(int) / 10**9
test['Datetime'] = pd.to_datetime(test['Datetime']).astype(int) / 10**9

In [9]:
# prompt: normalize all feature values.assume every column is numerical

from sklearn.preprocessing import MinMaxScaler

# Assuming 'train' and 'test' DataFrames are already defined and preprocessed

# Create a MinMaxScaler object
scaler = MinMaxScaler()

# Select numerical features (assuming all columns are numerical)
numerical_cols = train.select_dtypes(include=['number']).columns

# Fit the scaler on the training data
scaler.fit(train[numerical_cols])

# Transform both training and test data
train[numerical_cols] = scaler.transform(train[numerical_cols])
test[numerical_cols] = scaler.transform(test[numerical_cols])

In [10]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Convert 'Datetime' to numeric values


# Prepare training and testing data
X_train = train.drop(['TimeToArrival'], axis=1).values.astype('float32')
y_train = train['TimeToArrival'].values.astype('float32')
X_test = test.drop(['TimeToArrival'], axis=1).values.astype('float32')
y_test = test['TimeToArrival'].values.astype('float32')
# Check for NaN values in the original DataFrame
print("NaNs in original train DataFrame:\n", train.isna().sum())

# Feature scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Check for NaN values
print("NaNs in X_train:", pd.isna(X_train).any())
print("NaNs in y_train:", pd.isna(y_train).any())
print("NaNs in X_test:", pd.isna(X_test).any())
print("NaNs in y_test:", pd.isna(y_test).any())

# Reshape data for CNN
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

# Define the CNN model
model = Sequential()
model.add(Conv1D(32, 3, activation='relu', padding='same', input_shape=(X_train.shape[1], 1)))
model.add(MaxPooling1D(2))
model.add(Conv1D(64, 2, activation='relu', padding='same'))  # Use padding to prevent dimension issues
model.add(Flatten())
model.add(Dense(128, activation='relu'))
model.add(Dense(1))  # Output layer for regression

# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error')

# Implement early stopping
early_stopping = EarlyStopping(monitor='loss', patience=5, restore_best_weights=True)

# Train the model
history = model.fit(X_train, y_train, epochs=20, batch_size=32, callbacks=[early_stopping], verbose=1)

# Evaluate the model
loss = model.evaluate(X_test, y_test)
print("Mean Squared Error on Test Set:", loss)

# Make predictions
predictions = model.predict(X_test)

# Check for NaN predictions
print("Predictions:", predictions)
print("NaNs in predictions:", pd.isna(predictions).any())


NaNs in original train DataFrame:
 Datetime             0
DirectionRef         0
PublishedLineName    0
NextStopPointName    0
TimeToArrival        0
dtype: int64
NaNs in X_train: False
NaNs in y_train: False
NaNs in X_test: False
NaNs in y_test: False
Epoch 1/20


/usr/local/lib/python3.10/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


2/2 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - loss: 0.0748
Epoch 2/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0123  
Epoch 3/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0123 
Epoch 4/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0211 
Epoch 5/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0160 
Epoch 6/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0073
Epoch 7/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0081 
Epoch 8/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0084 
Epoch 9/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0083 
Epoch 10/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0067
Epoch 11/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0060
Epoch 12/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0061 
Epoch 13/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0057 
Epoch 14/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0055
Epoch 15/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0054
Epoch 16/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/

In [11]:
for i in range(len(predictions)):
  print("Predicted:", predictions[i][0], "Actual:", y_test[i])

Predicted: 0.05666222 Actual: 0.05927835
Predicted: 0.69016206 Actual: 1.1829897
Predicted: 0.08316589 Actual: 0.14690721
Predicted: 0.98003376 Actual: 1.5747423
Predicted: 0.10466353 Actual: 0.010309278
Predicted: 0.11837555 Actual: 0.17783505
Predicted: 0.09031529 Actual: 0.025773196
Predicted: 0.7800224 Actual: 0.23969072
Predicted: 0.077802196 Actual: 0.030927835
Predicted: 0.3076886 Actual: 0.28865978


In [12]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = model.predict(X_test)

# Calculate evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)

# Print the results
print('Mean Absolute Error:', mae)
print('Mean Squared Error:', mse)
print('Root Mean Squared Error:', rmse)
print('R-squared:', r2)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
Mean Absolute Error: 0.19784844
Mean Squared Error: 0.09117486
Root Mean Squared Error: 0.30195177
R-squared: 0.6600363850593567


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


In [13]:
model.save('cnn_model.h5')
